# Data Cleaning
## Logistics Dataset

Objective: Clean raw logistics data - handle duplicates, missing values, data types, dates, outliers.

## Import Libraries

In [5]:
import pandas as pd
import numpy as np
import sqlite3
import os
import re
import warnings
warnings.filterwarnings('ignore')

## Load Raw Data

In [6]:
import sqlite3
conn = sqlite3.connect('../database/data.sqlite')
df = pd.read_sql_query("SELECT * FROM shipments", conn)
print('Shape:', df.shape)
df.head(3)

Shape: (400000, 20)


,shipment_id,order_id,customer_name,origin_city,destination_city,origin_country,destination_country,ship_date,delivery_date,weight_kg,volume_m3,shipping_mode,status,base_cost_usd,surcharge_usd,total_cost_usd,currency,payment_terms,created_at,last_update
0,SHP-288390,53737,nguyen kim,Phnom Penh,Saigon,TH,Cambodia,"Oct 15, 2021","Jan 14, 2021",301.9,2.497,rail,LOST,1259.14,726.46,2305.8,THB,PrePaid,2023-02-21T13:50:23Z,2023-07-06T05:55:19Z
1,74871,382273,Vann Lee,Bangkok,Phnom Penh,Cambodia,VN,05/14/2022,08/08/2021,269.53,1.862,TRK,DELIVERED,"99,0",free,NaN,NaN,cod,2023-10-11 00:25:04,2023-10-02T09:42:53Z
2,918399,ORD-399488,Dara Smith,Bangkok,Phnompenh,NaN,Vietnam,04/03/2021,2020-11-18,488.99,2.648,trk,DELIVERED,849.16,1080.9,error,usd,cod,29/10/2023 20:10,2023-05-07 08:10:51


## Helper: Clean Numeric Values

In [7]:
def clean_numeric_val(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return val
    s = str(val).strip()
    if s.lower() in ('unknown', 'error', 'free', 'n/a', 'none', 'null', '', '-'):
        return np.nan
    s = s.replace('$', '').replace(' ', '')
    if ',' in s and '.' in s:
        if s.rindex(',') < s.rindex('.'):
            s = s.replace(',', '')
        else:
            s = s.replace('.', '').replace(',', '.')
    elif ',' in s:
        s = s.replace(',', '.')
    try:
        return float(s)
    except ValueError:
        return np.nan

## Step 1: Remove Duplicates

In [8]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
removed = before - after
pct = (removed / before) * 100 if before > 0 else 0
print(f'Before: {before}')
print(f'After: {after}')
print(f'Duplicates removed: {removed}')
print(f'Percentage removed: {pct:.2f}%')

Before: 400000
After: 400000
Duplicates removed: 0
Percentage removed: 0.00%


## Step 2: Convert Numeric Columns

In [9]:
object_cols = df.select_dtypes(include='object').columns
for col in object_cols:
    first_200 = df[col].head(200)
    converted = first_200.apply(clean_numeric_val)
    convertible = converted.notna().sum()
    rate = convertible / len(first_200) * 100
    if rate >= 60:
        df[col] = df[col].apply(clean_numeric_val)
        print(f'Converted {col} to numeric (conv rate: {rate:.1f}%)')

if 'weight_kg' in df.columns:
    neg = (df['weight_kg'] < 0).sum()
    if neg > 0:
        df['weight_kg'] = df['weight_kg'].abs()
        print(f'Absolute value applied to weight_kg ({neg} negative values)')

Converted shipment_id to numeric (conv rate: 66.5%)
Converted weight_kg to numeric (conv rate: 92.0%)
Converted base_cost_usd to numeric (conv rate: 99.0%)
Converted surcharge_usd to numeric (conv rate: 96.5%)
Converted total_cost_usd to numeric (conv rate: 63.0%)
Absolute value applied to weight_kg (11343 negative values)


## Step 3: Standardize Dates

In [10]:
import dateutil.parser

def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    try:
        return pd.Timestamp(dateutil.parser.parse(str(val)))
    except Exception:
        return pd.NaT

date_cols = [c for c in df.columns if 'date' in c.lower()]
for col in date_cols:
    before_na = df[col].isna().sum()
    df[col] = df[col].apply(parse_date)
    after_na = df[col].isna().sum()
    unparseable = after_na - before_na
    print(f'{col}: {unparseable} unparseable values converted to NaT')

ship_date: 3813 unparseable values converted to NaT
delivery_date: 3476 unparseable values converted to NaT
last_update: 8110 unparseable values converted to NaT


## Step 4: Handle Missing Values

In [11]:
for col in df.columns:
    if df[col].isna().sum() == 0:
        continue
    if pd.api.types.is_datetime64_any_dtype(df[col]):
        continue
    if pd.api.types.is_numeric_dtype(df[col]):
        med = df[col].median()
        df[col] = df[col].fillna(med)
        print(f'{col}: imputed {df[col].isna().sum()} missing with median {med:.2f}')
    else:
        mode_val = df[col].mode().iloc[0] if not df[col].mode().empty else 'unknown'
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: imputed with mode ({mode_val})')

shipment_id: imputed 0 missing with median 500334.00
order_id: imputed with mode (ORD-224932)
customer_name: imputed with mode (Sok Lee)
origin_city: imputed with mode (BKK)
destination_city: imputed with mode (PP)
origin_country: imputed with mode (VN)
destination_country: imputed with mode (KHM)
weight_kg: imputed 0 missing with median 247.34
volume_m3: imputed 0 missing with median 2.00
shipping_mode: imputed with mode (air)
status: imputed with mode (in transit)
base_cost_usd: imputed 0 missing with median 1002.65
surcharge_usd: imputed 0 missing with median 1001.23
total_cost_usd: imputed 0 missing with median 1705.37
currency: imputed with mode (USD)
payment_terms: imputed with mode (COD)
last_update: imputed with mode (2023-01-26 09:43:00)


## Step 5: Handle Outliers (IQR Capping)

In [12]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before_cap = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower, upper)
    if before_cap > 0:
        print(f'{col}: capped {before_cap} outliers to [{lower:.2f}, {upper:.2f}]')

shipment_id: capped 44538 outliers to [89105.62, 911916.62]
weight_kg: capped 7512 outliers to [-223.49, 717.74]
volume_m3: capped 3274 outliers to [-0.36, 4.36]
total_cost_usd: capped 77608 outliers to [710.65, 2679.18]


## Cleaning Summary

In [13]:
before = 400000
after = len(df)
loss = before - after
pct = (loss / before) * 100
print(f'Rows before: {before}')
print(f'Rows after: {after}')
print(f'Rows removed: {loss}')
print(f'Percentage retained: {100 - pct:.2f}%')
if pct > 10:
    print('WARNING: More than 10% of data was removed during cleaning.')

Rows before: 400000
Rows after: 400000
Rows removed: 0
Percentage retained: 100.00%


## Save Cleaned Data

In [14]:
# CSV
df.to_csv('../data/final_data.csv', index=False)
print('Saved to ../data/final_data.csv')

# SQLite — convert all values to native Python types for compatibility
df_sql = df.copy()
for col in df_sql.columns:
    if pd.api.types.is_datetime64_any_dtype(df_sql[col]):
        df_sql[col] = df_sql[col].apply(lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else None)
    elif pd.api.types.is_numeric_dtype(df_sql[col]):
        df_sql[col] = df_sql[col].apply(lambda x: None if pd.isna(x) else float(x))
    else:
        df_sql[col] = df_sql[col].apply(lambda x: None if pd.isna(x) else str(x))

db_path = '../data/logistics_clean.db'
conn = sqlite3.connect(db_path)
df_sql.to_sql('clean_shipments', conn, if_exists='replace', index=False)
conn.close()
print(f'Saved to {db_path} table clean_shipments ({len(df)} rows)')

Saved to ../data/final_data.csv
Saved to ../data/logistics_clean.db table clean_shipments (400000 rows)


## Verify in SQLite

In [15]:
conn = sqlite3.connect('../data/logistics_clean.db')
cur = conn.cursor()
cur.execute('SELECT COUNT(*) FROM clean_shipments')
print('Row count:', cur.fetchone()[0])
print('\nSample rows:')
cur.execute('SELECT * FROM clean_shipments LIMIT 10')
for row in cur.fetchall():
    print(row)
conn.close()

Row count: 400000

Sample rows:
(500334.0, '53737', '  nguyen kim  ', 'Phnom Penh', 'Saigon', 'TH', 'Cambodia', '2021-10-15 00:00:00', '2021-01-14 00:00:00', 301.9, 2.497, 'rail', 'LOST', 1259.14, 726.46, 2305.8, 'THB', 'PrePaid', '2023-02-21T13:50:23Z', '2023-07-06 05:55:19+00:00')
(89105.625, '382273', 'Vann Lee', 'Bangkok', 'Phnom Penh', 'Cambodia', 'VN', '2022-05-14 00:00:00', '2021-08-08 00:00:00', 269.53, 1.862, 'TRK', 'DELIVERED', 99.0, 1001.23, 1705.37, 'USD', 'cod', '2023-10-11 00:25:04', '2023-10-02 09:42:53+00:00')
(911916.625, 'ORD-399488', 'Dara Smith', 'Bangkok', 'Phnompenh', 'VN', 'Vietnam', '2021-04-03 00:00:00', '2020-11-18 00:00:00', 488.99, 2.648, 'trk', 'DELIVERED', 849.16, 1080.9, 1705.37, 'usd', 'cod', '29/10/2023 20:10', '2023-05-07 08:10:51')
(500334.0, 'ORD-37150', 'Nguyen Lee', 'HCMC', 'HCMC', 'Thailand', 'VN', '2024-08-10 00:00:00', '2020-09-05 00:00:00', 54.72, 3.523, 'Air Freight', 'INTRANSIT', 155.39, 1712.65, 1868.04, 'USD', 'prepaid', '2023-04-20 14:36:0

## Conclusion

The cleaning pipeline successfully:
- Removed duplicate records
- Converted numeric columns with mixed formatting (currency, European decimals)
- Standardized date columns to datetime
- Imputed missing values (median for numeric, mode for categorical)
- Capped outliers using the IQR method
- Exported cleaned data to CSV and SQLite

The final dataset is ready for analysis and modeling.